<div align="center">

# Data Projects and Hackathon 3  
## Project 
Sergio Fernandez, Alessandro Mecchia 

</div>

In [ ]:
import pandas as pd
import json
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import seaborn as sns
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import numpy as np
from transformers import pipeline
import requests
import time
from pathlib import Path
import subprocess
import sys
import time

try:
    import pyarrow as pa
    import pyarrow.json as paj
    import pyarrow.parquet as pq
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow"])
    import pyarrow as pa
    import pyarrow.json as paj
    import pyarrow.parquet as pq
    
import pyarrow.parquet as pq

import pyarrow.parquet as pq
import ipywidgets as widgets
from IPython.display import display


## Functions 

In [ ]:
def pct(x): 
    return f"{x:,} ({x/n*100:.1f}%)"

def is_empty(val):
    if val is None:
        return True
    if isinstance(val, str):
        return val == ""
    if isinstance(val, (list, np.ndarray)):
        return len(val) == 0
    return False

In [ ]:
def get_lang_from_openalex(doi):
    if not doi or doi == "":
        return None
    
    url = f"https://api.openalex.org/works/https://doi.org/{doi}"
    headers = {"User-Agent": "DBLP-lang-imputation/1.0 (your@email.com)"}
    
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code == 200:
            data = r.json()
            return data.get("language", None) 
        elif r.status_code == 429:
            time.sleep(5)   
            return get_lang_from_openalex(doi)
    except Exception:
        return None
    return None

In [ ]:
import json

def impute_with_cache(df, col, fetch_fn, cache_path):
    cache_path = Path(cache_path)
    mask = df[col].apply(is_empty) & ~df["doi"].apply(is_empty)
    missing_df = df[mask]

    if cache_path.exists():
        with open(cache_path, "r") as f:
            recovered = {int(k): v for k, v in json.load(f).items()}
        print(f"Cache found: loaded {len(recovered)} values from {cache_path}")
    else:
        recovered = {}
        for i, (idx, row) in enumerate(missing_df.iterrows()):
            value = fetch_fn(row["doi"])
            if value:
                recovered[idx] = value

            if (i + 1) % 10 == 0:
                print(f"  [{i+1}/{len(missing_df)}] recovered until now: {len(recovered)}")

            time.sleep(0.1)

        with open(cache_path, "w") as f:
            json.dump(recovered, f)
        print(f"Cache saved to {cache_path}")

    # Assign row by row to avoid issues with list values
    for idx, value in recovered.items():
        df.at[idx, col] = value

    print(f"Imputation completed:")
    print(f"Missing '{col}' before: {mask.sum()} and after: {df[col].apply(is_empty).sum()}")

In [ ]:
import json

def impute_with_cache(df, col, fetch_fn, cache_path):
    cache_path = Path(cache_path)
    mask = df[col].apply(is_empty) & ~df["doi"].apply(is_empty)
    missing_df = df[mask]

    if cache_path.exists():
        with open(cache_path, "r") as f:
            recovered = {int(k): v for k, v in json.load(f).items()}
        print(f"Cache found: loaded {len(recovered)} values from {cache_path}")
    else:
        recovered = {}
        for i, (idx, row) in enumerate(missing_df.iterrows()):
            value = fetch_fn(row["doi"])
            if value:
                recovered[idx] = value

            if (i + 1) % 10 == 0:
                print(f"  [{i+1}/{len(missing_df)}] recovered until now: {len(recovered)}")

            time.sleep(0.1)

        with open(cache_path, "w") as f:
            json.dump(recovered, f)
        print(f"Cache saved to {cache_path}")

    for idx, value in recovered.items():
        df.at[idx, col] = value

    print(f"Imputation completed:")
    print(f"Missing '{col}' before: {mask.sum()} and after: {df[col].apply(is_empty).sum()}")

In [ ]:
def get_lang_from_openalex(doi):
    if not doi or doi == "":
        return None
    try:
        r = requests.get(
            f"https://api.openalex.org/works/https://doi.org/{doi}",
            headers={"User-Agent": "DBLP-imputation/1.0"},
            timeout=10
        )
        if r.status_code != 200:
            return None
        return r.json().get("language", None)
    except Exception:
        return None


def get_keywords_from_openalex(doi):
    if not doi or doi == "":
        return None
    try:
        r = requests.get(
            f"https://api.openalex.org/works/https://doi.org/{doi}",
            headers={"User-Agent": "DBLP-imputation/1.0"},
            timeout=10
        )
        if r.status_code != 200:
            return None
        concepts = r.json().get("concepts", [])
        keywords = [
            c["display_name"] for c in concepts
            if c.get("level", 0) >= 1 and c.get("score", 0) >= 0.3
        ]
        return keywords if keywords else None
    except Exception:
        return None

## Data Creation 

da eseguire una sola volta 

In [ ]:
DATA_DIR = Path("data")
SOURCE_PATH = DATA_DIR / "DBLP-Citation-network-V18.jsonl"
TARGET_PATH = DATA_DIR / "DBLP-Citation-network-V18.parquet"
BLOCK_SIZE = 64 * 1024 * 1024  

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"File non trovato: {SOURCE_PATH}")

if pa.Codec.is_available("zstd"):
    COMPRESSION = "zstd"
elif pa.Codec.is_available("snappy"):
    COMPRESSION = "snappy"
else:
    COMPRESSION = None

print(f"Input : {SOURCE_PATH} ({SOURCE_PATH.stat().st_size / 1024**3:.2f} GiB)")
print(f"Output: {TARGET_PATH}")
print(f"Compressione: {COMPRESSION}")
print(f"Block size: {BLOCK_SIZE / 1024**2:.0f} MiB")

In [ ]:
if TARGET_PATH.exists():
    print(f"Parquet already exists, skipping conversion.")
else:
    reader = paj.open_json(
        SOURCE_PATH,
        read_options=paj.ReadOptions(block_size=BLOCK_SIZE),
    )

    writer = None
    rows_written = 0
    batches_written = 0
    started_at = time.perf_counter()

    try:
        while True:
            try:
                batch = reader.read_next_batch()
            except StopIteration:
                break

            if writer is None:
                writer = pq.ParquetWriter(
                    TARGET_PATH,
                    batch.schema,
                    compression=COMPRESSION,
                )

            writer.write_batch(batch)
            rows_written += batch.num_rows
            batches_written += 1

            if batches_written % 25 == 0:
                elapsed = time.perf_counter() - started_at
                print(f"Batch: {batches_written:>5} | Rows: {rows_written:>12,} | Elapsed: {elapsed:>8.1f}s")

        if writer is None:
            raise RuntimeError("JSONL file seems empty: no batch read.")
    finally:
        reader.close()
        if writer is not None:
            writer.close()

    elapsed = time.perf_counter() - started_at
    print(f"Conversion completed in {elapsed:.1f}s")
    print(f"Rows written : {rows_written:,}")
    print(f"JSONL size   : {SOURCE_PATH.stat().st_size / 1024**3:.2f} GiB")
    print(f"Parquet size : {TARGET_PATH.stat().st_size / 1024**3:.2f} GiB")

## Data Exploration 

In [ ]:
pf = pq.ParquetFile(r"data\DBLP-Citation-network-V18.parquet")

print(f"Total rows   : {pf.metadata.num_rows:,}")
print(f"Columns      : {pf.metadata.num_columns}")
print(f"Row groups   : {pf.metadata.num_row_groups}")

In [ ]:
for i, name in enumerate(pf.schema_arrow.names):
    print(f"{i}. {name}")

In [ ]:
pf = pq.ParquetFile(r"data\DBLP-Citation-network-V18.parquet")
columns = pf.schema_arrow.names

dropdown = widgets.Dropdown(options=columns, description="Colonna:")
output = widgets.Output()

def on_change(change):
    if change["type"] == "change" and change["name"] == "value":
        with output:
            output.clear_output()
            batch = next(pf.iter_batches(batch_size=10, columns=[change["new"]]))
            df = batch.to_pandas()
            display(df[change["new"]])

dropdown.observe(on_change)
display(dropdown, output)

In [ ]:
batch = next(pf.iter_batches(batch_size=1))
paper = batch.to_pandas().iloc[0]

for col, val in paper.items():
    print(f"{col:15}: {val}")

### Missing values 

In [ ]:
batch = next(pf.iter_batches(batch_size=10_000))
df = batch.to_pandas()
n = len(df)

# Simple columns
for col in ["id", "title", "abstract", "year", "page_start", "page_end",
            "lang", "volume", "issue", "issn", "isbn", "doi", "venue", "doc_type"]:
    missing = df[col].apply(is_empty).sum()
    print(f"  {col:15}: {pct(missing)}")

# Lists
print()
for col in ["keywords", "references", "url"]:
    missing = df[col].apply(is_empty).sum()
    print(f"  {col:15}: {pct(missing)}")

# Authors 
print()
authors_flat = pd.DataFrame(df["authors"].explode().dropna().tolist())
total_authors = len(authors_flat)
for col in ["id", "name", "org"]:
    missing = authors_flat[col].apply(is_empty).sum()
    print(f"  authors.{col:10}: {missing:,} ({missing/total_authors*100:.1f}% of autors)")

spiegare che ci interessa imputare solo le variabili che pensiamo possano essere utili per predirre il numero di citazioni o se un paper cita un'altro, ma prima potrebbe essere utile analizzare la distribuzione nei anni dei missing value.

In [ ]:
print(f"Min year: {df['year'].min()}")
print(f"Max year: {df['year'].max()}")

In [ ]:
papers = df.sort_values("year").reset_index(drop=True)
current = [0]

def show_paper(idx):
    paper = papers.iloc[idx]
    print(f"Paper {idx+1} / {len(papers)}")
    print(f"{'─'*60}")
    for col, val in paper.items():
        print(f"  {col:15}: {val}")

prev_btn = widgets.Button(description="Prev")
next_btn = widgets.Button(description="Next")
out = widgets.Output()

def on_prev(_):
    if current[0] > 0:
        current[0] -= 1
    with out:
        out.clear_output()
        show_paper(current[0])

def on_next(_):
    if current[0] < len(papers) - 1:
        current[0] += 1
    with out:
        out.clear_output()
        show_paper(current[0])

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)

with out:
    show_paper(current[0])

display(widgets.HBox([prev_btn, next_btn]), out)

come possiamo notare i primi due paper hanno delle date strane, per sicurezza abbiamo deciso di compararli usando il url e vedere se combaciano, dopo aver controllato i primi 15 abbiamo visto che i primi due avevano la data sbagliata

In [ ]:
sorted_idx = df["year"].sort_values().index

df.loc[sorted_idx[0], "year"] = 2006
df.loc[sorted_idx[1], "year"] = 1976

print(f"Min year: {df['year'].min()}")
print(f"Max year: {df['year'].max()}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sns.histplot(data=df, x="year", bins=100, ax=ax)
ax.set_title("Distribution of papers by year")
ax.set_xlabel("Year")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

#### 1. Lang emputation

spiegare che se avessivo avuto accesso ha una parte del paper avremmo potuto usare un modello dei trasformer per analizzare la scrittura del paper e imputare in modo corretto la lingua del paper, siccome non è il caso possiamo provare altri metodi. 

Prima di tutto dobbiamo vedere le possibilità o comunque i valori che abbiamo presenti, in caso fossero tutti inglesi allora li mettiamo inglesi tutti. 

In [ ]:
lang_counts = df["lang"].value_counts(dropna=False)
lang_counts.index = lang_counts.index.fillna("(null)")
print(f"Unique values of 'lang' (sample of {n:,} rows):\n")
print(lang_counts.to_string())

siccome non è il caso e abbiamo diverse possibilità, la prossima cosa che possiamo fare è utilizzare il doi e cercare in un'altra libbreria se riusciamo a identificare la lingua del paper

In [ ]:
impute_with_cache(df, "lang",     get_lang_from_openalex,     "data/cache_lang.json")

abbiamo ancora 80 valori mancanti, la prossima possibilità è quella di usare l'abstract e con un modello trasnformer rilevare che lingua è e imputarla, ma prima di tutto dobbiamo vedere se per caso la lingua del abstract non è la stessa a quella inseria in lang

In [ ]:
sample_by_lang = (
    df[~df["abstract"].apply(is_empty) & ~df["lang"].apply(is_empty)]
    .groupby("lang")
    .apply(lambda x: x.sample(1, random_state=42))
    .reset_index(drop=True)
    [["lang", "title", "abstract"]]
    .sort_values("lang")
)

for _, row in sample_by_lang.iterrows():
    print(f"{'─'*60}")
    print(f"LANG : {row['lang']}")
    print(f"TITLE: {row['title']}")
    print(f"ABSTRACT: {row['abstract'][:300]}...")
    print()

ora che abbiamo visto che è fattibile applichiamo il metodo con il modello

In [ ]:
from transformers import pipeline

lang_detector = pipeline(
    "text-classification",
    model="papluca/xlm-roberta-base-language-detection"
)

mask = df["lang"].apply(is_empty) & ~df["abstract"].apply(is_empty)

df.loc[mask, "lang"] = df.loc[mask, "abstract"].apply(
    lambda x: lang_detector(x[:512], truncation=True)[0]["label"]
)

print(f"Imputed: {mask.sum()} | Still missing: {df['lang'].apply(is_empty).sum()}")

come possiamo notare rimangono ancora 50 valori mancanti dovuto al fatto che molti non hanno l'abstract, ma comunque siamo risuciti a imputare un totale di 242 su 292 

#### 2. Keywords imputation

come primo tentativo possiamo vedere se con il doi e usando onealex possiamo ottenere delle keywords 

In [ ]:
impute_with_cache(df, "keywords", get_keywords_from_openalex, "data/cache_keywords.json")

siamo riusciti ad imputare un totale di circa 500 valori, i rimanenti sono dovuti a richieste che ritornano un errore, o a mancanti DOI, per il resto possiamo generarli partendo dall'absatract

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained("fabiochiu/t5-base-tag-generation")
model = AutoModelForSeq2SeqLM.from_pretrained("fabiochiu/t5-base-tag-generation")

def extract_keywords_from_abstract(abstract):
    if not abstract or abstract == "":
        return None
    
    try:
        # Tokenize input
        inputs = tokenizer.encode(abstract[:512], return_tensors="pt", max_length=512, truncation=True)
        
        # Generate output
        outputs = model.generate(inputs, max_length=50, num_beams=4, early_stopping=True)
        
        # Decode result
        result = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Split and clean keywords
        keywords = [kw.strip() for kw in result.split(",") if kw.strip()]
        return keywords if keywords else None
    except Exception as e:
        print(f"Error processing abstract: {e}")
        return None


mask = df["keywords"].apply(is_empty) & ~df["abstract"].apply(is_empty)
print(f"Records to impute with transformer: {mask.sum()}")

for idx, row in df[mask].iterrows():
    keywords = extract_keywords_from_abstract(row["abstract"])
    if keywords:
        df.at[idx, "keywords"] = keywords

print(f"Still missing: {df['keywords'].apply(is_empty).sum()}")

## Data Visualizzation 